In [7]:
# Set up imports and autoreload for development.
from material_hasher.hasher.bawl import BAWLHasher
from material_hasher.similarity import PymatgenStructureSimilarity

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
# Load the LeMat-Bulk dataset from Hugging Face Datasets
from datasets import load_dataset

dataset = load_dataset("LeMaterial/LeMat-Bulk", "compatible_pbe", split="train")

In [4]:
# Utility function to convert a dataset row into a pymatgen Structure object
from pymatgen.core import Structure


def get_structure_from_hf_row(row):
    """Get a pymatgen Structure from a dictionary.
    The dictionary should contain the following keys:
        - lattice_vectors: list of lists containing the lattice vectors
        - species_at_sites: list of species at each site
        - cartesian_site_positions: list of cartesian site positions

    Parameters
    ----------
    row : dict
        Dictionary containing the structure information.

    Returns
    -------
    pymatgen.Structure
        Pymatgen Structure object.
    """

    return Structure(
        lattice=[x for y in row["lattice_vectors"] for x in y],
        species=row["species_at_sites"],
        coords=row["cartesian_site_positions"],
        coords_are_cartesian=True,
    )

In [10]:
# Preprocess LeMat-Bulk dataset to create composition-based indexing for efficient structure matching
composition_lematbulk = dataset.select_columns(["chemical_formula_reduced", "elements"]).to_pandas()

from collections import Counter

# preprocess every row of elements to have a string that allows for faster comparison afterwards
composition_lematbulk["elements_signature"] = composition_lematbulk["elements"].apply(lambda x: frozenset(Counter(x).items()))
signature_to_indices = (
    composition_lematbulk
    .groupby("elements_signature")
    .apply(lambda df: df.index.tolist())
    .to_dict()
)


In [ ]:
import tqdm
import requests
# Download MatPES dataset from AWS S3 and decompress the gzipped JSON file
url = "https://s3.us-east-1.amazonaws.com/materialsproject-contribs/MatPES_2025_1/MatPES-PBE-2025.1.json.gz"

path_matpes = "matpes_dataset.json.gz"

r = requests.get(url, stream=True)

total_size = int(r.headers.get("content-length", 0))
block_size = 1024


with (
    open(path_matpes, "wb") as file,
    tqdm.tqdm(total=total_size, unit="iB", unit_scale=True) as pbar,
):
    for data in r.iter_content(block_size):
        file.write(data)
        pbar.update(len(data))


# uncompress the file
import gzip
import shutil

with gzip.open(path_matpes, "rb") as f_in:
    with open("matpes_dataset.json", "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)


In [8]:
import ase
from ase.db import connect

class ASEDB:
    """Wrapper class for ASE Database files.

    Parameters
    ----------
    path : str
        Path to the ASE Database file
    """

    def __init__(self, path: str, **kwargs):
        self.path = path
        self.db = connect(path, **kwargs)

    def write(self, atoms: ase.Atoms, attributes: dict = None):
        self.db.write(atoms, data=attributes, natom=len(atoms))

    def batch_write(self, atoms: list[ase.Atoms], attributes: list[dict] = None):
        with connect(self.path, append=True) as db:
            for atom, attribute in zip(atoms, attributes):
                db.write(atom, data=attribute)

    def __len__(self):
        return len(self.db)

    def get_node_counts(self) -> list[int]:
        return [atom.get("natom") for atom in self.db.select()]

    def __getitem__(self, idx: int | slice):
        if isinstance(idx, slice):
            return list(
                self.db.select(
                    [("id", "<=", idx.stop), ("id", ">=", idx.start)],
                )
            )
        else:
            return next(self.db.select(id=idx + 1))

    def close(self):
        self.db.close()


In [ ]:
### THIS CELL IS USED TO CREATE A MATPES DATABASE FROM THE MATPES JSON FILE ###

from pymatgen.io.ase import AseAtomsAdaptor
import orjson
import numpy as np
import pandas as pd
import tqdm

path_matpes = "matpes_dataset.json"


with open(path_matpes, "rb") as f:
    matpes_data = orjson.loads(f.read())

all_entries = []
atoms_buffer = []
batch_size = 1000
asedb = ASEDB(path="matpes_dataset.db")

for row in (pbar := tqdm.tqdm(matpes_data, desc="Processing items")):
    structure = Structure.from_dict(row["structure"])
    all_entries.append(AseAtomsAdaptor.get_atoms(structure))
    natoms = len(structure)

    # forces
    f_tmp = np.array(row["forces"], dtype="float64")

    entry = {
        "immutable_id": row["matpes_id"],
        "trajectory_id": row["provenance"]["original_mp_id"],
        "id": row["matpes_id"],
        "relaxation_step": row["provenance"]["md_step"],
        "relaxation_number": 0,
        "energy": row["energy"],
        "nsites": natoms,
        "chemical_formula_descriptive": structure.composition.formula,
        "max_force": np.linalg.norm(row["forces"], axis=1).max(),
    }
    all_entries.append(entry)

    if len(atoms_buffer) >= batch_size:
        asedb.batch_write(atoms_buffer, [None for _ in atoms_buffer])
        atoms_buffer = []

    pbar.update(1)

pbar.close()

if len(atoms_buffer) > 0:
    asedb.batch_write(atoms_buffer, [None for _ in atoms_buffer])

df_mp = pd.DataFrame(all_entries)
df_mp.to_csv("matpes_dataset.csv", index=False)


In [ ]:
import tqdm
from collections import defaultdict
import pandas as pd

from pymatgen.io.ase import AseAtomsAdaptor

similarity_scores = defaultdict(list)
db = ASEDB("matpes_dataset.db")

df_matpes = pd.read_csv("matpes_dataset.csv")

similarity = PymatgenStructureSimilarity()

for i in tqdm.tqdm(range(len(db))):
    atoms_row = db[i].toatoms()
    structure = AseAtomsAdaptor.get_structure(atoms_row)

    elements = [str(e) for e in structure.composition.elements]
    target_signature = frozenset(Counter(elements).items())
    keep_lemat_bulk_indices = signature_to_indices[target_signature]

    for idx in keep_lemat_bulk_indices:
        lemat_bulk_row = dataset[idx]
        lemat_bulk_structure = get_structure_from_hf_row(lemat_bulk_row)

        similarity_score = similarity.get_similarity_score(structure, lemat_bulk_structure)

        if similarity_score > 0.0:
            similarity_scores[df_matpes.iloc[i].id].append((similarity_score, lemat_bulk_row["immutable_id"]))


In [ ]:
import tqdm
import pandas as pd
from collections import defaultdict, Counter
from pymatgen.io.ase import AseAtomsAdaptor
from multiprocessing import Pool, Manager, cpu_count

# Assumed definitions
# - ASEDB, get_structure_from_hf_row, PymatgenStructureSimilarity must already be defined/imported
db = ASEDB("matpes_dataset.db")
df_matpes = pd.read_csv("matpes_dataset.csv")

similarity = PymatgenStructureSimilarity()

# Precompute target signature index map (signature_to_indices)
# signature_to_indices = ... (already defined earlier)

# Number of processes
NUM_WORKERS = max(cpu_count() - 1, 1)

# Split indices into chunks
def chunkify(lst, n):
    return [lst[i::n] for i in range(n)]

def worker(db_indices):
    local_scores = defaultdict(list)
    for i in db_indices:
        atoms_row = db[i].toatoms()
        structure = AseAtomsAdaptor.get_structure(atoms_row)

        elements = [str(e) for e in structure.composition.elements]
        target_signature = frozenset(Counter(elements).items())

        keep_lemat_bulk_indices = signature_to_indices.get(target_signature, [])

        for idx in keep_lemat_bulk_indices:
            lemat_bulk_row = dataset[idx]
            lemat_bulk_structure = get_structure_from_hf_row(lemat_bulk_row)

            similarity_score = similarity.get_similarity_score(structure, lemat_bulk_structure)

            if similarity_score > 0.0:
                local_scores[df_matpes.iloc[i].id].append(
                    (similarity_score, lemat_bulk_row["immutable_id"])
                )
    return local_scores

# Parallel execution
all_indices = list(range(100))
index_chunks = chunkify(all_indices, NUM_WORKERS)

with Pool(processes=NUM_WORKERS) as pool:
    results = list(tqdm.tqdm(pool.imap(worker, index_chunks), total=NUM_WORKERS))

# Merge results
similarity_scores = defaultdict(list)
for res in results:
    for k, v in res.items():
        similarity_scores[k].extend(v)
